In [24]:
import os
import pandas as pd 

In [25]:
dir_path = "C:\\Users\\berkk\\Desktop\\Bitirme_Projesi\\MR"

listdir = os.listdir(dir_path)
listdir

['OAS30001_MR_d0129',
 'OAS30001_MR_d0757',
 'OAS30001_MR_d2430',
 'OAS30001_MR_d3132',
 'OAS30001_MR_d3746',
 'OAS30001_MR_d4467',
 'OAS30002_MR_d0371',
 'OAS30002_MR_d0653',
 'OAS30002_MR_d1680',
 'OAS30002_MR_d2340',
 'OAS30002_MR_d2345',
 'OAS30003_MR_d0558',
 'OAS30003_MR_d1631',
 'OAS30003_MR_d2669',
 'OAS30003_MR_d2682',
 'OAS30003_MR_d3320',
 'OAS30003_MR_d3731',
 'OAS30003_MR_d4954',
 'OAS30004_MR_d1101',
 'OAS30004_MR_d2229',
 'OAS30004_MR_d2232',
 'OAS30004_MR_d3457',
 'OAS30004_MR_d4700',
 'OAS30005_MR_d0143',
 'OAS30005_MR_d0581',
 'OAS30005_MR_d1274',
 'OAS30005_MR_d2377',
 'OAS30005_MR_d2384',
 'OAS30005_MR_d3453',
 'OAS30005_MR_d4620',
 'OAS30006_MR_d0166',
 'OAS30006_MR_d0373',
 'OAS30006_MR_d1308',
 'OAS30006_MR_d2341',
 'OAS30006_MR_d2342',
 'OAS30006_MR_d3386',
 'OAS30006_MR_d4440',
 'OAS30007_MR_d0061',
 'OAS30007_MR_d1636',
 'OAS30007_MR_d1641',
 'OAS30007_MR_d1981',
 'OAS30007_MR_d2722',
 'OAS30007_MR_d3807',
 'OAS30008_MR_d0061',
 'OAS30008_MR_d1313',
 'OAS30008

In [26]:

subject_id = list(map(lambda x:x.split("_")[0],listdir))
session_num = list(map(lambda x:int(x.split("_")[2][1:]),listdir))
session_num[:10]

[129, 757, 2430, 3132, 3746, 4467, 371, 653, 1680, 2340]

In [27]:
mr = pd.DataFrame( 
    {
        "id":listdir,
        "OASISID": subject_id,
        "session_num":session_num
    }

)
mr.head(10)

,id,OASISID,session_num
0,OAS30001_MR_d0129,OAS30001,129
1,OAS30001_MR_d0757,OAS30001,757
2,OAS30001_MR_d2430,OAS30001,2430
3,OAS30001_MR_d3132,OAS30001,3132
4,OAS30001_MR_d3746,OAS30001,3746
5,OAS30001_MR_d4467,OAS30001,4467
6,OAS30002_MR_d0371,OAS30002,371
7,OAS30002_MR_d0653,OAS30002,653
8,OAS30002_MR_d1680,OAS30002,1680
9,OAS30002_MR_d2340,OAS30002,2340


In [28]:
data_csv = pd.read_csv("4-OASIS3_UDSb4_cdr.csv")
cdr_score = data_csv[["OASISID","days_to_visit","memory","age at visit"]].copy()
cdr_score["days_to_visit"] = cdr_score["days_to_visit"].astype(int)

cdr_score.head(16)

,OASISID,days_to_visit,memory,age at visit
0,OAS30001,0,0.0,65.19
1,OAS30001,339,0.0,66.12
2,OAS30001,722,0.0,67.17
3,OAS30001,1106,0.0,68.22
4,OAS30001,1456,0.0,69.18
5,OAS30001,1894,0.0,70.38
6,OAS30001,2181,0.0,71.17
7,OAS30001,2699,0.0,72.59
8,OAS30001,3025,0.0,73.48
9,OAS30001,3332,0.0,74.32


In [29]:
results = []
for _,cdr_row in cdr_score.iterrows():
    cdr_id = cdr_row["OASISID"]
    cdr_days = cdr_row["days_to_visit"]
    cdr_memory = cdr_row["memory"]
    cdr_age = cdr_row["age at visit"]

    print(f"id:{cdr_id}--days:{cdr_days}--memory{cdr_memory}--age{cdr_age}")

    mr_subset = mr[mr["OASISID"]== cdr_id]
    if len(mr_subset)==0:
        results.append({
                "OASISID":cdr_id,
                "days_to_visit":cdr_days,
                "memory":cdr_memory,
                "age at visit":cdr_age,
                "path":None,
                "session_num":None
            })
    else:
        hold = float('inf')
        day = None
        for i in mr_subset["session_num"]:     
            if abs(cdr_days-i) < hold:
                if abs(cdr_days-i)<365:
                    hold = abs(cdr_days-i)
                    day = i
                else:
                    continue
        path_series = mr_subset.loc[mr_subset["session_num"] == day, "id"]

        if not path_series.empty:
            path_value = path_series.iloc[0]
            session_value = day
        else:
            path_value = None
            session_value = None

        results.append({
            "OASISID": cdr_id,
            "days_to_visit": cdr_days,
            "memory": cdr_memory,
            "age at visit": cdr_age,
            "path": path_value,
            "session_num": session_value
        })


df_merged = pd.DataFrame(results)

df_merged.head(10)

id:OAS30001--days:0--memory0.0--age65.19
id:OAS30001--days:339--memory0.0--age66.12
id:OAS30001--days:722--memory0.0--age67.17
id:OAS30001--days:1106--memory0.0--age68.22
id:OAS30001--days:1456--memory0.0--age69.18
id:OAS30001--days:1894--memory0.0--age70.38
id:OAS30001--days:2181--memory0.0--age71.17
id:OAS30001--days:2699--memory0.0--age72.59
id:OAS30001--days:3025--memory0.0--age73.48
id:OAS30001--days:3332--memory0.0--age74.32
id:OAS30001--days:3675--memory0.0--age75.26
id:OAS30001--days:3977--memory0.0--age76.09
id:OAS30001--days:4349--memory0.0--age77.11
id:OAS30001--days:4823--memory0.0--age78.41
id:OAS30001--days:5184--memory0.0--age79.4
id:OAS30002--days:0--memory0.0--age67.25
id:OAS30002--days:751--memory2.0--age69.31
id:OAS30002--days:1169--memory0.0--age70.45
id:OAS30002--days:1508--memory0.0--age71.38
id:OAS30002--days:1850--memory0.0--age72.32
id:OAS30002--days:2263--memory0.0--age73.45
id:OAS30002--days:2585--memory0.0--age74.33
id:OAS30002--days:2961--memory0.0--age75.3

,OASISID,days_to_visit,memory,age at visit,path,session_num
0,OAS30001,0,0.0,65.19,OAS30001_MR_d0129,129.0
1,OAS30001,339,0.0,66.12,OAS30001_MR_d0129,129.0
2,OAS30001,722,0.0,67.17,OAS30001_MR_d0757,757.0
3,OAS30001,1106,0.0,68.22,OAS30001_MR_d0757,757.0
4,OAS30001,1456,0.0,69.18,None,NaN
5,OAS30001,1894,0.0,70.38,None,NaN
6,OAS30001,2181,0.0,71.17,OAS30001_MR_d2430,2430.0
7,OAS30001,2699,0.0,72.59,OAS30001_MR_d2430,2430.0
8,OAS30001,3025,0.0,73.48,OAS30001_MR_d3132,3132.0
9,OAS30001,3332,0.0,74.32,OAS30001_MR_d3132,3132.0


In [30]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8626 entries, 0 to 8625
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   OASISID        8626 non-null   object 
 1   days_to_visit  8626 non-null   int64  
 2   memory         8620 non-null   float64
 3   age at visit   8626 non-null   float64
 4   path           3845 non-null   object 
 5   session_num    3845 non-null   float64
dtypes: float64(3), int64(1), object(2)
memory usage: 404.5+ KB


In [31]:
df_merged["path"].isnull().sum()

np.int64(4781)

In [32]:
df_merged.to_csv("mr.csv",index=False)

In [33]:
df = df_merged.dropna(subset=["path"])
df.to_csv("dropna.csv", index=False)
df.head(15)

,OASISID,days_to_visit,memory,age at visit,path,session_num
0,OAS30001,0,0.0,65.19,OAS30001_MR_d0129,129.0
1,OAS30001,339,0.0,66.12,OAS30001_MR_d0129,129.0
2,OAS30001,722,0.0,67.17,OAS30001_MR_d0757,757.0
3,OAS30001,1106,0.0,68.22,OAS30001_MR_d0757,757.0
6,OAS30001,2181,0.0,71.17,OAS30001_MR_d2430,2430.0
7,OAS30001,2699,0.0,72.59,OAS30001_MR_d2430,2430.0
8,OAS30001,3025,0.0,73.48,OAS30001_MR_d3132,3132.0
9,OAS30001,3332,0.0,74.32,OAS30001_MR_d3132,3132.0
10,OAS30001,3675,0.0,75.26,OAS30001_MR_d3746,3746.0
11,OAS30001,3977,0.0,76.09,OAS30001_MR_d3746,3746.0


In [34]:
dataset=  df.copy()
data = dataset[["memory","path","session_num"]]

data.head(10)


,memory,path,session_num
0,0.0,OAS30001_MR_d0129,129.0
1,0.0,OAS30001_MR_d0129,129.0
2,0.0,OAS30001_MR_d0757,757.0
3,0.0,OAS30001_MR_d0757,757.0
6,0.0,OAS30001_MR_d2430,2430.0
7,0.0,OAS30001_MR_d2430,2430.0
8,0.0,OAS30001_MR_d3132,3132.0
9,0.0,OAS30001_MR_d3132,3132.0
10,0.0,OAS30001_MR_d3746,3746.0
11,0.0,OAS30001_MR_d3746,3746.0


In [35]:
mr_uniq = mr["id"].unique()
mr_uniq

array(['OAS30001_MR_d0129', 'OAS30001_MR_d0757', 'OAS30001_MR_d2430', ...,
       'OAS31472_MR_d0045', 'OAS31473_MR_d0136', 'OAS31474_MR_d0150'],
      shape=(2835,), dtype=object)

In [36]:
data = mr[~mr["id"].isin(dataset["path"])] # 365 de yakınında veri olmayan mr görüntüleri
print(data)

                     id   OASISID  session_num
6     OAS30002_MR_d0371  OAS30002          371
11    OAS30003_MR_d0558  OAS30003          558
12    OAS30003_MR_d1631  OAS30003         1631
20    OAS30004_MR_d2232  OAS30004         2232
24    OAS30005_MR_d0581  OAS30005          581
...                 ...       ...          ...
2706  OAS31338_MR_d0120  OAS31338          120
2737  OAS31367_MR_d0156  OAS31367          156
2772  OAS31404_MR_d0095  OAS31404           95
2818  OAS31457_MR_d0114  OAS31457          114
2829  OAS31469_MR_d0106  OAS31469          106

[232 rows x 3 columns]


In [37]:
unique = dataset["path"].unique()
unique


array(['OAS30001_MR_d0129', 'OAS30001_MR_d0757', 'OAS30001_MR_d2430', ...,
       'OAS31472_MR_d0045', 'OAS31473_MR_d0136', 'OAS31474_MR_d0150'],
      shape=(2603,), dtype=object)

In [38]:
arr_conflict = []
for i in unique:
    data_subset = dataset[dataset["path"]==i][["days_to_visit","memory","session_num"]]
    uniq_subset = data_subset["memory"].unique()
    if len(uniq_subset)>1:
        arr_conflict.append(data_subset)


In [39]:
len(arr_conflict)

203

In [40]:

# Conflict'leri temizle ve en iyi match'leri bul
cleaned_data = []

for i in arr_conflict:
    sNum = i["session_num"].iloc[0]  # Session number'ı al
    hold = float('inf')
    best_memory = None
    best_days = None
    
    for _, j in i.iterrows():
        value = abs(j["days_to_visit"] - sNum)
        if hold > value:
            hold = value
            best_memory = j["memory"]
            best_days = j["days_to_visit"]
    
   
    cleaned_data.append({
        'session_num': sNum,
        'best_memory': best_memory,
        'best_days': best_days,
        'temporal_diff': hold
    })

print(f"Temizlenen conflict sayısı: {len(cleaned_data)}")


for idx, data in enumerate(cleaned_data[:5]):
    print(f"Conflict {idx+1}: memory={data['best_memory']}, temporal_diff={data['temporal_diff']} gün")

Temizlenen conflict sayısı: 203
Conflict 1: memory=0.5, temporal_diff=51.0 gün
Conflict 2: memory=0.5, temporal_diff=177.0 gün
Conflict 3: memory=0.5, temporal_diff=102.0 gün
Conflict 4: memory=0.0, temporal_diff=127.0 gün
Conflict 5: memory=0.5, temporal_diff=84.0 gün


In [41]:
# Temizlenmiş verileri dataset'e uygula
for clean in cleaned_data:
    sNum = clean['session_num']
    best_mem = clean['best_memory']
    best_days = clean['best_days']
    
    # Bu session_num'a sahip tüm satırları bul
    mask = dataset['session_num'] == sNum
    
    # Sadece en iyi match'i tut, diğerlerini sil
    dataset = dataset[~(mask & (dataset['days_to_visit'] != best_days))]

print(f"\nConflict temizleme sonrası kayıt sayısı: {len(dataset)}")
print(f"Memory dağılımı:")
print(dataset['memory'].value_counts().sort_index())

# Temizlenmiş veriyi kaydet
dataset.to_csv("5-mr_path.csv", index=False)
print("\n✓ Temizlenmiş veri 'mr_path1.csv' olarak kaydedildi")


Conflict temizleme sonrası kayıt sayısı: 3427
Memory dağılımı:
memory
0.0    2764
0.5     415
1.0     222
2.0      26
Name: count, dtype: int64

✓ Temizlenmiş veri 'mr_path1.csv' olarak kaydedildi


In [42]:

final_df = pd.read_csv("5-mr_path.csv")
print(f"5-mr_path.csv dosyasındaki kayıt sayısı: {len(final_df)}")
print(f"Başlangıç dataset kayıt sayısı: {len(dataset)}")
print(f"Silinen duplicate sayısı: {len(dataset) - len(final_df)}")

print(f"\nMemory değer dağılımı (temizlenmiş):")
print(final_df['memory'].value_counts().sort_index())

print(f"\nDuplicate kontrol (temizlenmiş veride):")
path_counts = final_df['path'].value_counts()
duplicates = path_counts[path_counts > 1]
print(f"Duplicate path sayısı: {len(duplicates)}")

if len(duplicates) > 0:
    print("Kalan duplicate'ler:")
    print(duplicates.head())
else:
    print("✓ Tüm duplicate'ler temizlenmiş!")

5-mr_path.csv dosyasındaki kayıt sayısı: 3427
Başlangıç dataset kayıt sayısı: 3427
Silinen duplicate sayısı: 0

Memory değer dağılımı (temizlenmiş):
memory
0.0    2764
0.5     415
1.0     222
2.0      26
Name: count, dtype: int64

Duplicate kontrol (temizlenmiş veride):
Duplicate path sayısı: 873
Kalan duplicate'ler:
path
OAS30232_MR_d2324    3
OAS30117_MR_d5243    3
OAS31028_MR_d1285    3
OAS30127_MR_d0837    3
OAS30049_MR_d1633    3
Name: count, dtype: int64


In [43]:
arr_conflict = []
for i in unique:
    data_subset = final_df[final_df["path"]==i][["days_to_visit","memory","session_num"]]
    uniq_subset = data_subset["memory"].unique()
    if len(uniq_subset)>1:
        arr_conflict.append(data_subset)


In [44]:
len(arr_conflict)

0